# Тема 5. Кластеризація мовою Python

Ви відкрили Jupyter Notebook: файл, у якому пояснення чергуються з короткими програмами мовою Python. Його програми групують імпульси радарів без підказок людини, як на рис. 5.3 теми 5.

**Як працювати.** Jupyter Notebook складається з комірок: в одних текст, як у цій, в інших програма. Клацніть першу комірку з програмою, вона стоїть одразу під цим текстом, і натисніть Shift+Enter: комірка виконається, а під нею з’явиться результат. Так само виконуйте решту комірок по черзі, згори донизу.

In [ ]:
# Набір імпульсів зі сторінки теми 5. Програма читає його як таблицю чисел.
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

SET = 1                               # номер набору імпульсів: 1, 2 або 3
NAME = f"tema-5-nabir-{SET}.csv"
ADDRESS = "https://korotkiyeugene.github.io/rozumna-syla-rer-chapter/t5/" + NAME
if not os.path.exists(NAME):          # файла ще немає поруч: програма бере його зі сторінки теми
    urllib.request.urlretrieve(ADDRESS, NAME)

table = np.loadtxt(NAME, delimiter=",", skiprows=1)
time, frequency, width, direction = table.T      # стовпці файлу: мс, МГц, мкс, градуси


def uk(number):
    """Число з десятковою комою і щонайбільше двома знаками після неї."""
    return f"{number:.2f}".rstrip("0").rstrip(".").replace(".", ",")


print("Імпульсів у наборі:", len(table))
print("Перший імпульс: час приходу", uk(time[0]), "мс, частота", uk(frequency[0]), "МГц,",
      "тривалість", uk(width[0]), "мкс, напрямок", uk(direction[0]), "градусів")

## Крок 1. Кожен імпульс стає точкою

Файл набору містить PDW: один рядок на кожен імпульс. Стовпці файлу такі: `час_мс`, `частота_МГц`, `тривалість_мкс`, `напрямок_град`. Наступна програма позначає кожен імпульс точкою. По горизонталі відкладено частоту, по вертикалі тривалість імпульсу.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(frequency, width, color="gray")
plt.xlabel("частота, МГц")
plt.ylabel("тривалість імпульсу, мкс")
plt.title(f"Набір {SET}: кожна точка позначає один імпульс")
plt.grid(alpha=0.3)
plt.show()

## Крок 2. Програма шукає групи

Дві точки програма вважає сусідами, якщо вони близькі і за частотою, і за тривалістю імпульсу. Наскільки близькі, задають два числа: `NEAR_F` для частоти і `NEAR_WIDTH` для тривалості. Програма ділить різницю частот двох точок на `NEAR_F`, а різницю тривалостей на `NEAR_WIDTH`. Якщо сума квадратів цих двох чисел не більша за 1, точки є сусідами.

Групу утворюють точки, з’єднані ланцюжком сусідів. Якщо в ланцюжку менше точок, ніж задає число `LEAST`, групи немає: такі точки лишаються без групи.

In [ ]:
NEAR_F = 14.0         # наскільки можуть різнитися частоти сусідів, МГц
NEAR_WIDTH = 0.14     # наскільки можуть різнитися тривалості сусідів, мкс
LEAST = 4             # найменша кількість точок у групі


def clusters(frequency, width):
    """Номер групи кожної точки, від 1. Нуль означає, що точка лишилася без групи."""
    count = len(frequency)
    df = (frequency[:, None] - frequency[None, :]) / NEAR_F
    dw = (width[:, None] - width[None, :]) / NEAR_WIDTH
    near = df ** 2 + dw ** 2 <= 1.0                 # near[i, j]: чи є точки i та j сусідами
    chain = np.full(count, -1)                      # номер ланцюжка кожної точки
    chains = 0
    for start in range(count):
        if chain[start] >= 0:
            continue
        chain[start] = chains
        queue = [start]
        while queue:                                # програма йде від сусіда до сусіда
            i = queue.pop()
            for j in np.flatnonzero(near[i] & (chain < 0)):
                chain[j] = chains
                queue.append(j)
        chains += 1
    kept = [k for k in range(chains) if (chain == k).sum() >= LEAST]
    kept.sort(key=lambda k: frequency[chain == k].mean())      # групи нумеруються за частотою
    group = np.zeros(count, dtype=int)
    for number, k in enumerate(kept, 1):
        group[chain == k] = number
    return group


group = clusters(frequency, width)
print("Груп:", group.max())
print("Точок без групи:", (group == 0).sum())

## Крок 3. Групи на рисунку і в таблиці

Наступна програма фарбує кожну групу у свій колір. Під рисунком вона друкує таблицю груп: скільки в групі імпульсів, їхню середню частоту, середню тривалість і межі напрямків.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(frequency[group == 0], width[group == 0], facecolors="none", edgecolors="gray", label="без групи")
for number in range(1, group.max() + 1):
    plt.scatter(frequency[group == number], width[group == number], label=f"група {number}")
plt.xlabel("частота, МГц")
plt.ylabel("тривалість імпульсу, мкс")
plt.title(f"Набір {SET}: групи, які знайшла програма")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

for number in range(1, group.max() + 1):
    mine = group == number
    print(f"Група {number}: імпульсів {mine.sum()}, частота {uk(frequency[mine].mean())} МГц,",
          f"тривалість {uk(width[mine].mean())} мкс,",
          f"напрямок від {direction[mine].min():.0f}° до {direction[mine].max():.0f}°")

## Завдання

Після кожної зміни виконуйте програми знову, згори донизу.

1. Набір 1 записано від трьох радарів. Скільки груп знайшла програма? Замініть рядок `NEAR_F = 14.0` на `NEAR_F = 2.0`. Потім поставте `NEAR_F = 80.0` і `NEAR_WIDTH = 0.8`. Що сталося з групами в кожному випадку? Поверніть початкові числа.
2. У першій програмі замініть `SET = 1` на `SET = 2`. У наборі 2 теж три радари. Скільки груп знайшла програма? Подивіться на напрямки в таблиці груп: яка група видає, що в ній два радари? Доберіть `NEAR_F`, з яким груп стає три.
3. Поверніть `NEAR_F = 14.0` і відкрийте набір 3: `SET = 3`. Скільки груп знайшла програма? Насправді радарів у цьому наборі два. Порівняйте напрямки й тривалості груп. Які групи належать одному радару? Чим відрізняються імпульси цього радара?
4. Змініть `LEAST` і подивіться, що сталося з точками без групи.
5. У короткому звіті запишіть, як числа `NEAR_F`, `NEAR_WIDTH` і `LEAST` впливають на результат. Поясніть, чому результат програми перевіряє людина.